In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


RANDOM_STATE = 42

sns.set_theme(style="whitegrid")

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATASET = "FD002"  # Change to FD001, FD002, FD003, or FD004

DATA_PATH = PROJECT_ROOT / "data" / "raw" / f"train_{DATASET}.txt"
RESULTS_PATH = PROJECT_ROOT / "results" / "figures" / DATASET

RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Data file:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

In [ ]:
index_columns = [
    "engine_id",
    "cycle",
]

setting_columns = [
    "setting_1",
    "setting_2",
    "setting_3",
]

sensor_columns = [
    f"sensor_{number}" for number in range(1, 22)
]

column_names = index_columns + setting_columns + sensor_columns

print(column_names)
print("Number of columns:", len(column_names))

In [ ]:
train = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None,
    names=column_names
)

print(train.head())
print("\nDataset shape:", train.shape)

In [ ]:
print("Number of engines:", train["engine_id"].nunique())
print("Number of rows:", len(train))
print("Missing values:", train.isna().sum().sum())
print("Duplicated rows:", train.duplicated().sum())

print("\nData types:")
print(train.dtypes)

print("\nSummary:")
print(train.describe().T)

In [ ]:
maximum_cycles = (
    train.groupby("engine_id")["cycle"]
    .max()
    .rename("maximum_cycle")
)

train = train.merge(
    maximum_cycles,
    on="engine_id",
    how="left",
    validate="many_to_one"
)

train["RUL"] = train["maximum_cycle"] - train["cycle"]

print(
    train[
        ["engine_id", "cycle", "maximum_cycle", "RUL"]
    ].head(10)
)

In [ ]:
print(
    train.loc[
        train["engine_id"] == 1,
        ["engine_id", "cycle", "maximum_cycle", "RUL"]
    ].tail()
)

In [ ]:
assert train["RUL"].min() == 0
assert (train["RUL"] >= 0).all()
assert train["engine_id"].nunique() > 0
assert train.isna().sum().sum() == 0

final_rul = train.groupby("engine_id")["RUL"].last()
assert (final_rul == 0).all()

print("All RUL checks passed.")

In [ ]:
engine_lifetimes = train.groupby("engine_id")["cycle"].max()

plt.figure(figsize=(12, 5))
plt.bar(
    engine_lifetimes.index,
    engine_lifetimes.values,
    color="steelblue"
)

plt.xlabel("Engine ID")
plt.ylabel("Number of operating cycles")
plt.title(f"Operating Lifetime of {DATASET} Training Engines")
plt.tight_layout()

plt.savefig(
    RESULTS_PATH / "engine_lifetimes.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
engine_number = 1

engine_data = train[
    train["engine_id"] == engine_number
].copy()

selected_sensors = [
    "sensor_2",
    "sensor_7",
    "sensor_11",
    "sensor_12",
]

fig, axes = plt.subplots(
    len(selected_sensors),
    1,
    figsize=(10, 10),
    sharex=True
)

for axis, sensor in zip(axes, selected_sensors):
    axis.plot(
        engine_data["cycle"],
        engine_data[sensor],
        color="darkred"
    )

    axis.set_ylabel(sensor)
    axis.set_title(f"{sensor} for Engine {engine_number}")

axes[-1].set_xlabel("Operating cycle")

plt.tight_layout()

plt.savefig(
    RESULTS_PATH / "engine_1_sensor_trends.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
sensor_statistics = train[sensor_columns].agg(
    ["mean", "std", "min", "max"]
).T

constant_sensors = sensor_statistics[
    sensor_statistics["std"] < 1e-8
].index.tolist()

print("Constant sensors:", constant_sensors)
print(sensor_statistics)

In [ ]:
usable_sensors = [
    sensor
    for sensor in sensor_columns
    if sensor not in constant_sensors
]

print("Usable sensors:", usable_sensors)
print("Number of usable sensors:", len(usable_sensors))

In [ ]:
feature_columns = (
    ["cycle"]
    + setting_columns
    + usable_sensors
)

X = train[feature_columns].copy()
y = train["RUL"].copy()
groups = train["engine_id"].copy()

print("Feature matrix:", X.shape)
print("Target:", y.shape)

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_indices, validation_indices = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_indices]
X_validation = X.iloc[validation_indices]

y_train = y.iloc[train_indices]
y_validation = y.iloc[validation_indices]

training_engines = set(groups.iloc[train_indices])
validation_engines = set(groups.iloc[validation_indices])

print("Training engines:", len(training_engines))
print("Validation engines:", len(validation_engines))
print("Shared engines:", training_engines & validation_engines)

assert training_engines.isdisjoint(validation_engines)

In [ ]:
def evaluate_model(name, y_true, y_prediction):
    """Calculate regression performance metrics."""

    mae = mean_absolute_error(y_true, y_prediction)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_prediction)
    )
    r_squared = r2_score(y_true, y_prediction)

    return {
        "model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r_squared,
    }

In [ ]:
mean_training_rul = y_train.mean()

baseline_predictions = np.full(
    shape=len(y_validation),
    fill_value=mean_training_rul
)

results = [
    evaluate_model(
        "Mean RUL baseline",
        y_validation,
        baseline_predictions
    )
]

pd.DataFrame(results)

In [ ]:
linear_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "regressor",
            LinearRegression()
        ),
    ]
)

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_validation)

results.append(
    evaluate_model(
        "Linear regression",
        y_validation,
        linear_predictions
    )
)

pd.DataFrame(results).round(3)

In [ ]:
random_forest = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=15,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=RANDOM_STATE
            )
        ),
    ]
)

random_forest.fit(X_train, y_train)

forest_predictions = random_forest.predict(X_validation)

results.append(
    evaluate_model(
        "Random Forest",
        y_validation,
        forest_predictions
    )
)

results_table = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

print(results_table.round(3))

In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_validation,
    forest_predictions,
    alpha=0.25,
    s=15,
    color="teal"
)

minimum_value = min(
    y_validation.min(),
    forest_predictions.min()
)

maximum_value = max(
    y_validation.max(),
    forest_predictions.max()
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--",
    color="black",
    label="Perfect prediction"
)

plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.title("Random Forest: Actual vs Predicted RUL")
plt.legend()
plt.tight_layout()

plt.savefig(
    RESULTS_PATH / "random_forest_predictions.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
metrics_path = PROJECT_ROOT / "results" / "metrics"
metrics_path.mkdir(parents=True, exist_ok=True)

results_table.to_csv(
    metrics_path / f"{DATASET.lower()}_initial_model_results.csv",
    index=False
)

print("Results saved to:", metrics_path)

In [ ]:
results_plot = results_table.melt(
    id_vars="model",
    value_vars=["MAE", "RMSE"],
    var_name="Metric",
    value_name="Error"
)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=results_plot,
    x="model",
    y="Error",
    hue="Metric"
)

plt.xlabel("Model")
plt.ylabel("Prediction error in cycles")
plt.title("Validation Performance of RUL Models")
plt.xticks(rotation=15)
plt.tight_layout()

plt.savefig(
    RESULTS_PATH / "model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()